# Exploration des données – Condition de valve hydraulique

**Objectif** : Classification binaire — la valve est-elle en état optimal (100%) ou non ?

**Source** : UCI – Condition Monitoring of Hydraulic Systems  
**Capteurs utilisés** : PS2 (pression, 100 Hz) et FS1 (débit, 10 Hz)  
**Variable cible** : `profile.txt` colonne index 1 — valve condition

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
print('Librairies chargées.')

## 1. Chargement des données brutes

In [ ]:
ps2     = pd.read_csv('../data/raw/PS2.txt', sep='\t', header=None)
fs1     = pd.read_csv('../data/raw/FS1.txt', sep='\t', header=None)
profile = pd.read_csv('../data/raw/profile.txt', sep='\t', header=None)

print(f'PS2  : {ps2.shape}  → {ps2.shape[1]} pts/cycle à 100 Hz (60 s)')
print(f'FS1  : {fs1.shape}  → {fs1.shape[1]} pts/cycle à 10 Hz (60 s)')
print(f'Profile : {profile.shape}')

## 2. Variable cible – Valve condition

Colonne index 1 du fichier `profile.txt` :
- `100` → optimal switching behavior → **classe 1**
- `90` → small lag → classe 0
- `80` → severe lag → classe 0
- `73` → close to total failure → classe 0

In [ ]:
valve_raw = profile.iloc[:, 1]
target    = (valve_raw == 100).astype(int)

print('Distribution valve condition (valeurs brutes):')
print(valve_raw.value_counts().sort_index())
print(f'\nClasse 0 (non optimal) : {(target == 0).sum()} cycles ({(target == 0).mean()*100:.1f}%)')
print(f'Classe 1 (optimal)     : {(target == 1).sum()} cycles ({(target == 1).mean()*100:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution brute
valve_counts = valve_raw.value_counts().sort_index()
axes[0].bar([str(v) for v in valve_counts.index], valve_counts.values,
            color=['#e74c3c', '#e67e22', '#f39c12', '#2ecc71'])
axes[0].set_title('Distribution brute – Valve condition')
axes[0].set_xlabel('Valeur')
axes[0].set_ylabel('Nombre de cycles')
for i, v in enumerate(valve_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Distribution binaire
binary_counts = target.value_counts().sort_index()
axes[1].bar(['Non optimal (0)', 'Optimal (1)'], binary_counts.values,
            color=['#e74c3c', '#2ecc71'])
axes[1].set_title('Distribution binaire – Cible finale')
axes[1].set_ylabel('Nombre de cycles')
for i, v in enumerate(binary_counts.values):
    axes[1].text(i, v + 5, f'{v}\n({v/len(target)*100:.1f}%)', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print('\n→ Les classes sont quasi équilibrées (51% / 49%). Pas de déséquilibre majeur.')

## 3. Visualisation des signaux bruts

On compare un cycle **optimal** vs un cycle **non optimal** pour PS2 et FS1.

In [ ]:
idx_optimal     = target[target == 1].index[0]
idx_non_optimal = target[target == 0].index[0]

fig, axes = plt.subplots(2, 2, figsize=(15, 8))

# PS2 – optimal
t_ps2 = np.linspace(0, 60, ps2.shape[1])
axes[0, 0].plot(t_ps2, ps2.iloc[idx_optimal].values, color='#2ecc71', linewidth=0.5)
axes[0, 0].set_title(f'PS2 – Cycle optimal (idx {idx_optimal})', fontweight='bold')
axes[0, 0].set_xlabel('Temps (s)')
axes[0, 0].set_ylabel('Pression (bar)')

# PS2 – non optimal
axes[0, 1].plot(t_ps2, ps2.iloc[idx_non_optimal].values, color='#e74c3c', linewidth=0.5)
axes[0, 1].set_title(f'PS2 – Cycle non optimal (idx {idx_non_optimal})', fontweight='bold')
axes[0, 1].set_xlabel('Temps (s)')
axes[0, 1].set_ylabel('Pression (bar)')

# FS1 – optimal
t_fs1 = np.linspace(0, 60, fs1.shape[1])
axes[1, 0].plot(t_fs1, fs1.iloc[idx_optimal].values, color='#2ecc71', linewidth=1)
axes[1, 0].set_title(f'FS1 – Cycle optimal (idx {idx_optimal})', fontweight='bold')
axes[1, 0].set_xlabel('Temps (s)')
axes[1, 0].set_ylabel('Débit (L/min)')

# FS1 – non optimal
axes[1, 1].plot(t_fs1, fs1.iloc[idx_non_optimal].values, color='#e74c3c', linewidth=1)
axes[1, 1].set_title(f'FS1 – Cycle non optimal (idx {idx_non_optimal})', fontweight='bold')
axes[1, 1].set_xlabel('Temps (s)')
axes[1, 1].set_ylabel('Débit (L/min)')

plt.suptitle('Comparaison signaux bruts : optimal vs non optimal', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Feature Engineering

Chaque cycle est une série temporelle. On extrait des **statistiques résumées** pour créer des features exploitables par un algorithme de classification.

**Features extraites** (11 par capteur × 2 capteurs = 22 features) :
`mean`, `std`, `min`, `max`, `median`, `q25`, `q75`, `range`, `rms`, `skew`, `kurtosis`

In [ ]:
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test  = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze()

print(f'Train : {X_train.shape} | Test : {X_test.shape}')
print(f'Features : {list(X_train.columns)}')

## 5. Distribution des features par classe

In [ ]:
ps2_features = [c for c in X_train.columns if c.startswith('ps2')]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, feat in enumerate(ps2_features):
    ax = axes[i]
    ax.hist(X_train.loc[y_train == 0, feat], bins=40, alpha=0.6, label='Non optimal', color='#e74c3c', density=True)
    ax.hist(X_train.loc[y_train == 1, feat], bins=40, alpha=0.6, label='Optimal', color='#2ecc71', density=True)
    ax.set_title(feat, fontsize=9)
    ax.legend(fontsize=7)

# Masquer les axes en trop
for j in range(len(ps2_features), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution des features PS2 par classe (train set)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('→ Plusieurs features PS2 montrent des distributions très distinctes entre classes.')

In [ ]:
fs1_features = [c for c in X_train.columns if c.startswith('fs1')]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, feat in enumerate(fs1_features):
    ax = axes[i]
    ax.hist(X_train.loc[y_train == 0, feat], bins=40, alpha=0.6, label='Non optimal', color='#e74c3c', density=True)
    ax.hist(X_train.loc[y_train == 1, feat], bins=40, alpha=0.6, label='Optimal', color='#2ecc71', density=True)
    ax.set_title(feat, fontsize=9)
    ax.legend(fontsize=7)

for j in range(len(fs1_features), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution des features FS1 par classe (train set)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Matrice de corrélation

In [ ]:
corr = X_train.corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5, ax=ax,
            annot_kws={'size': 7})
ax.set_title('Matrice de corrélation des features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('→ Forte corrélation attendue entre mean/median/rms (redondance partielle).')
print('→ Le Random Forest gère bien ces corrélations, pas besoin de sélection de features.')

## 7. Séparabilité linéaire – PCA 2D

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(
    X_pca[:, 0], X_pca[:, 1],
    c=y_train, cmap='RdYlGn', alpha=0.6, s=15, edgecolors='none'
)
legend = ax.legend(*scatter.legend_elements(), labels=['Non optimal (0)', 'Optimal (1)'])
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title('PCA 2D – Séparabilité des classes (train set)', fontweight='bold')
plt.tight_layout()
plt.show()

total_var = pca.explained_variance_ratio_.sum()
print(f'Variance expliquée par PC1+PC2 : {total_var*100:.1f}%')
print('→ Les deux classes sont visuellement bien séparées : un modèle non-linéaire devrait exceller.')

## 8. Analyse temporelle – Stabilité des features sur les cycles

On vérifie que le signal ne dérive pas dans le temps (drift potentiel).

In [ ]:
all_features = pd.concat([X_train, X_test], ignore_index=True)
all_target   = pd.concat([y_train, y_test], ignore_index=True)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

window = 50
rolling_mean_ps2 = all_features['ps2_mean'].rolling(window).mean()
rolling_mean_fs1 = all_features['fs1_mean'].rolling(window).mean()

axes[0].plot(rolling_mean_ps2, color='#3498db', linewidth=1.2, label=f'Moyenne glissante ({window} cycles)')
axes[0].axvline(x=2000, color='red', linestyle='--', linewidth=1.5, label='Limite train/test')
axes[0].set_ylabel('PS2 mean (bar)')
axes[0].set_title('Évolution temporelle de PS2 mean sur tous les cycles', fontweight='bold')
axes[0].legend()

axes[1].plot(rolling_mean_fs1, color='#9b59b6', linewidth=1.2, label=f'Moyenne glissante ({window} cycles)')
axes[1].axvline(x=2000, color='red', linestyle='--', linewidth=1.5, label='Limite train/test')
axes[1].set_xlabel('Numéro de cycle')
axes[1].set_ylabel('FS1 mean (L/min)')
axes[1].set_title('Évolution temporelle de FS1 mean sur tous les cycles', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()
print('→ Pas de drift visible : les signaux sont stationnaires. Le split temporel fixe est fiable.')

## 9. Résultats du modèle

Les résultats détaillés sont dans `models/eval_metrics.json` après entraînement.

In [ ]:
import json
from pathlib import Path

eval_path = Path('../models/eval_metrics.json')
train_path = Path('../models/train_metrics.json')

if eval_path.exists() and train_path.exists():
    with open(train_path) as f:
        train_m = json.load(f)
    with open(eval_path) as f:
        eval_m = json.load(f)

    print('=== Résultats entraînement (cross-validation) ===')
    print(f"  Meilleur modèle  : {train_m['best_model']}")
    print(f"  CV F1-macro      : {train_m['cv_f1_macro']}")
    print(f"  Tous les modèles : {train_m['all_models_cv_f1']}")
    print()
    print('=== Résultats test set (cycles 2000–2204) ===')
    for k, v in eval_m.items():
        if k != 'confusion_matrix':
            print(f'  {k:<20} : {v}')

    # Matrice de confusion
    cm = np.array(eval_m['confusion_matrix'])
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Non optimal', 'Optimal'],
                yticklabels=['Non optimal', 'Optimal'])
    ax.set_xlabel('Prédit')
    ax.set_ylabel('Réel')
    ax.set_title('Matrice de confusion – Test set', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Modèle non encore entraîné. Lancer : ./run.sh pipeline")

## 10. Conclusions

| Choix | Justification |
|---|---|
| **Features statistiques** | Les séries temporelles de 6000/600 pts sont trop larges pour être données brutes à un classifieur tabular. Les stats résumées (mean, std, skew...) capturent l'essentiel du comportement du signal. |
| **Random Forest** | Robuste aux corrélations entre features, pas besoin de scaling, interprétable via feature importances, excellentes performances en CV. |
| **StratifiedKFold(5)** | Garantit que la proportion de classes est respectée dans chaque fold, essentiel pour une évaluation fiable du F1-macro. |
| **Split temporel fixe** | Les 2000 premiers cycles = train, le reste = test. Respecte la temporalité des données industrielles et évite le data leakage. |
| **F1-macro comme métrique** | Équilibre la performance sur les deux classes indépendamment de leur taille, plus robuste que l'accuracy seule. |